# Model Training (coupler_NCap_cap_matrix)

## Configuration

In [1]:
## the parameter file has the hyperparameters
## start there if you want to change the setup

from parameters import *

## Library

In [ ]:
import os, gc, joblib, json, time

## keep the tensorflow warnings down
## comment these out if you want the warnings back
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf 
tf.keras.backend.set_floatx("float32") ## make the backend use float32 which will be the same as the datahelps speed it up

from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16') ## makes it faster

from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Input, Dense, Activation, Dropout, LeakyReLU
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from keras_tuner import HyperModel, RandomSearch
from tensorflow.keras.models import load_model
from pathlib import Path

import numpy as np
import pandas as pd
## imports that come up later
from tensorflow.python.client import device_lib
import sys
import matplotlib.pyplot as plt
import math
import platform
from IPython.display import clear_output
from keras_tuner import BayesianOptimization
from mpl_toolkits.mplot3d import Axes3D
import csv
from datetime import datetime
from tensorflow.keras import Sequential

seed = 0

## if the seed stays the same, the random numbers stay the same too
## you get the same random values every run
np.random.seed(seed)

## set the tensorflow seed too
tf.random.set_seed(seed)

## Check GPU

In [ ]:

## check what hardware tensorflow can see
print(device_lib.list_local_devices())
## run !{sys.executable} m pip install u pip
## run !{sys.executable} m pip install u "tensorflow[andcuda]"
## in this cell and restart the kernel.

print(tf.config.list_physical_devices("GPU"))
## check the cuda packages
!{sys.executable} -m pip list | egrep "tensorflow|nvidia-(cuda|cudnn|cublas|nccl)"

## Dataset

### Load

In [ ]:
## load scaled (trainfit) data saved by the preprocessing notebook
## (these are "scaled", not truly "augmented")

def load_scaled_split(kind, split):
    return np.load(f"{DATA_DIR}/npy/{split}_{kind}_encoding_scaled.npy", allow_pickle=True)

if 'Try Both' in ENCODING_TYPE:
    X_train_one_hot_encoding = load_scaled_split('one_hot', 'x_train')
    X_val_one_hot_encoding   = load_scaled_split('one_hot', 'x_val')
    X_test_one_hot_encoding  = load_scaled_split('one_hot', 'x_test')

    y_train_one_hot_encoding = load_scaled_split('one_hot', 'y_train')
    y_val_one_hot_encoding   = load_scaled_split('one_hot', 'y_val')
    y_test_one_hot_encoding  = load_scaled_split('one_hot', 'y_test')

    X_train_linear_encoding = load_scaled_split('linear', 'x_train')
    X_val_linear_encoding   = load_scaled_split('linear', 'x_val')
    X_test_linear_encoding  = load_scaled_split('linear', 'x_test')

    y_train_linear_encoding = load_scaled_split('linear', 'y_train')
    y_val_linear_encoding   = load_scaled_split('linear', 'y_val')
    y_test_linear_encoding  = load_scaled_split('linear', 'y_test')

else:
    if 'one hot' in ENCODING_TYPE:
        kind = 'one_hot'
    elif 'Linear' in ENCODING_TYPE:
        kind = 'linear'
    else:
        raise ValueError(f"Unknown ENCODING_TYPE: {ENCODING_TYPE}")

    X_train = load_scaled_split(kind, 'x_train')
    X_val   = load_scaled_split(kind, 'x_val')
    X_test  = load_scaled_split(kind, 'x_test')

    y_train = load_scaled_split(kind, 'y_train')
    y_val   = load_scaled_split(kind, 'y_val')
    y_test  = load_scaled_split(kind, 'y_test')

## load separate continuous / finger_count arrays for multihead training (Option B)
if 'one hot' in ENCODING_TYPE or 'Try Both' in ENCODING_TYPE:
    def _load_part(split, part):
        return np.load(f"{DATA_DIR}/npy/y_{split}_one_hot_encoding_scaled_{part}.npy", allow_pickle=True)

    y_train_cont = _load_part('train', 'continuous')
    y_val_cont   = _load_part('val',   'continuous')
    y_test_cont  = _load_part('test',  'continuous')

    y_train_fc   = _load_part('train', 'fingers')
    y_val_fc     = _load_part('val',   'fingers')
    y_test_fc    = _load_part('test',  'fingers')

    headers_cont = np.load(str(Path(METADATA_DIR) / "y_columns_continuous.npy"), allow_pickle=True).astype(str).tolist()
    headers_fc   = np.load(str(Path(METADATA_DIR) / "y_columns_fingers.npy"),   allow_pickle=True).astype(str).tolist()
    n_continuous  = len(headers_cont)
    n_fc_classes  = len(headers_fc)
    print(f"Multi-head: {n_continuous} continuous outputs, {n_fc_classes} finger_count classes")

X_train = X_train.astype("float32")
y_train = y_train.astype("float32")
X_val   = X_val.astype("float32")
y_val   = y_val.astype("float32")
if 'one hot' in ENCODING_TYPE or 'Try Both' in ENCODING_TYPE:
    y_train_cont = y_train_cont.astype("float32")
    y_val_cont   = y_val_cont.astype("float32")
    y_test_cont  = y_test_cont.astype("float32")
    y_train_fc   = y_train_fc.astype("float32")
    y_val_fc     = y_val_fc.astype("float32")
    y_test_fc    = y_test_fc.astype("float32")

### Visualize

In [ ]:
## look at the train and test shapes

if 'Try Both' not in ENCODING_TYPE:
    print('X_train.shape: ', X_train.shape)
    print('X_val.shape: ', X_val.shape)
    print('y_train.shape: ', y_train.shape)
    print('y_val.shape: ', y_val.shape)
    print('y_train[0]: ', y_train[0])
else:
    print('X_train_linear_encoding.shape: ', X_train_linear_encoding.shape)
    print('X_val_linear_encoding.shape: ', X_val_linear_encoding.shape)
    print('y_train_linear_encoding.shape: ', y_train_linear_encoding.shape)
    print('y_val_linear_encoding.shape: ', y_val_linear_encoding.shape)
    print('y_train_linear_encoding[0]: ', y_train_linear_encoding[0])

    print('X_train_one_hot_encoding.shape: ', X_train_one_hot_encoding.shape)
    print('X_val_one_hot_encoding.shape: ', X_val_one_hot_encoding.shape)
    print('y_train_one_hot_encoding.shape: ', y_train_one_hot_encoding.shape)
    print('y_val_one_hot_encoding.shape: ', y_val_one_hot_encoding.shape)
    print('y_train_one_hot_encoding[0]: ', y_train_one_hot_encoding[0])
if 'Try Both' not in ENCODING_TYPE:
    ## should match the columns top_to_top, top_to_bottom, top_to_ground, bottom_to_bottom, bottom_to_ground, ground_to_ground
    display(X_train) ## can check this in previous script as well after loading to make sure it matches
else:
    display(X_train_one_hot_encoding)
    display(X_train_linear_encoding)
## check the split

if 'Try Both' not in ENCODING_TYPE:
    total = len(X_train) + len(X_test) + len(X_val)
    print('---------------------------------------')  
    print('Train set shape x:                {}, {:.2f}%'.format(len(X_train), (len(X_train)*100.)/total))
    print('Validation set shape x:           {}, {:.2f}%'.format(len(X_train), (len(X_val)*100.)/total))
    print('Test set shape x:                 {}, {:.2f}%'.format(len(X_test), (len(X_test)*100.)/total))
    print('---------------------------------------')

    total = len(y_train) + len(y_test) + len(y_val)
    print('---------------------------------------')  
    print('Train set shape y:                {}, {:.2f}%'.format(len(y_train), (len(y_train)*100.)/total))
    print('Validation set shape y:           {}, {:.2f}%'.format(len(y_val), (len(y_val)*100.)/total))
    print('Test set shape y:                 {}, {:.2f}%'.format(len(y_test), (len(y_test)*100.)/total))
    print('---------------------------------------')
else:
    total = len(X_train_one_hot_encoding) + len(X_test_one_hot_encoding) + len(X_val_one_hot_encoding)
    print('---------------------------------------')  
    print('Train set shape x one_hot_encoding:      {}, {:.2f}%'.format(len(X_train_one_hot_encoding), (len(X_train_one_hot_encoding)*100.)/total))
    print('Validation set shape x one_hot_encoding: {}, {:.2f}%'.format(len(X_train_one_hot_encoding), (len(X_val_one_hot_encoding)*100.)/total))
    print('Test set shape x one_hot_encoding:       {}, {:.2f}%'.format(len(X_test_one_hot_encoding), (len(X_test_one_hot_encoding)*100.)/total))
    print('---------------------------------------')

    total = len(y_train_one_hot_encoding) + len(y_test_one_hot_encoding) + len(y_val_one_hot_encoding)
    print('---------------------------------------')  
    print('Train set shape y one_hot_encoding:         {}, {:.2f}%'.format(len(y_train_one_hot_encoding), (len(y_train_one_hot_encoding)*100.)/total))
    print('Validation set shape y one_hot_encoding:    {}, {:.2f}%'.format(len(y_val_one_hot_encoding), (len(y_val_one_hot_encoding)*100.)/total))
    print('Test set shape y one_hot_encoding:          {}, {:.2f}%'.format(len(y_test_one_hot_encoding), (len(y_test_one_hot_encoding)*100.)/total))

    total = len(X_train_linear_encoding) + len(X_test_linear_encoding) + len(X_val_linear_encoding)
    print('---------------------------------------')  
    print('Train set shape x linear_encoding:      {}, {:.2f}%'.format(len(X_train_linear_encoding), (len(X_train_linear_encoding)*100.)/total))
    print('Validation set shape x linear_encoding: {}, {:.2f}%'.format(len(X_train_linear_encoding), (len(X_val_linear_encoding)*100.)/total))
    print('Test set shape x linear_encoding:       {}, {:.2f}%'.format(len(X_test_linear_encoding), (len(X_test_linear_encoding)*100.)/total))
    print('---------------------------------------')

    total = len(y_train_linear_encoding) + len(y_test_linear_encoding) + len(y_val_linear_encoding)
    print('---------------------------------------')  
    print('Train set shape y linear_encoding:         {}, {:.2f}%'.format(len(y_train_linear_encoding), (len(y_train_linear_encoding)*100.)/total))
    print('Validation set shape y linear_encoding:    {}, {:.2f}%'.format(len(y_val_linear_encoding), (len(y_val_linear_encoding)*100.)/total))
    print('Test set shape y linear_encoding:          {}, {:.2f}%'.format(len(y_test_linear_encoding), (len(y_test_linear_encoding)*100.)/total))
%matplotlib inline

## bin the data and look at how its distributed, probably the more random/spread out the better
## for training, but this will improve as the database fills out

if 'Try Both' not in ENCODING_TYPE:
    ## training set
    save_encoding = ENCODING_TYPE.replace(' ','_')
    
    num_cols = X_train.shape[1]
    num_rows = math.ceil(num_cols / 3)
    
    fig, axes = plt.subplots(num_rows, 3, figsize=(10, 3 * num_rows))
    axes = axes.ravel()
    
    with open(str(Path(METADATA_DIR) / 'X_names'), 'r') as f:
        column_labels = f.read().splitlines()
    
    for i in range(num_cols):
        axes[i].hist(X_train[:, i], bins=30, edgecolor='black')
        axes[i].set_title(f'Training Set {column_labels[i]}')
        axes[i].set_xlabel(f'{column_labels[i]}')
        axes[i].set_ylabel('Frequency')
    
    ## remove unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/training_set_data_distribution{save_encoding}.pdf')
    plt.show()

    ## validation set
    num_cols = X_val.shape[1]
    num_rows = math.ceil(num_cols / 3)
    
    fig, axes = plt.subplots(num_rows, 3, figsize=(10, 3 * num_rows))
    axes = axes.ravel()
    
    with open(str(Path(METADATA_DIR) / 'X_names'), 'r') as f:
        column_labels = f.read().splitlines()
    
    for i in range(num_cols):
        axes[i].hist(X_val[:, i], bins=30, edgecolor='black')
        axes[i].set_title(f'Validation Set {column_labels[i]}')
        axes[i].set_xlabel(f'{column_labels[i]}')
        axes[i].set_ylabel('Frequency')
    
    ## remove unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/validation_set_data_distribution{save_encoding}.pdf')
    plt.show()

    ## test set
    num_cols = X_test.shape[1]
    num_rows = math.ceil(num_cols / 3)
    
    fig, axes = plt.subplots(num_rows, 3, figsize=(10, 3 * num_rows))
    axes = axes.ravel()
    
    with open(str(Path(METADATA_DIR) / 'X_names'), 'r') as f:
        column_labels = f.read().splitlines()
    
    for i in range(num_cols):
        axes[i].hist(X_test[:, i], bins=30, edgecolor='black')
        axes[i].set_title(f'Test Set {column_labels[i]}')
        axes[i].set_xlabel(f'{column_labels[i]}')
        axes[i].set_ylabel('Frequency')
    
    ## remove unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/test_set_data_distribution{save_encoding}.pdf')
    plt.show()
    
else: ## just plot linear encoding for now to not get plot overwhelm
    ## training set
    num_cols = X_train_linear_encoding.shape[1]
    num_rows = math.ceil(num_cols / 3)
    
    fig, axes = plt.subplots(num_rows, 3, figsize=(10, 3 * num_rows))
    axes = axes.ravel()
    
    with open(str(Path(METADATA_DIR) / 'X_names'), 'r') as f:
        column_labels = f.read().splitlines()
    
    for i in range(num_cols):
        axes[i].hist(X_train_linear_encoding[:, i], bins=30, edgecolor='black')
        axes[i].set_title(f'Training Set {column_labels[i]}')
        axes[i].set_xlabel(f'{column_labels[i]}')
        axes[i].set_ylabel('Frequency')
    
    ## remove unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    plt.savefig(str(Path(PLOTS_DIR) / 'training_set_data_distribution_linear_encoding.pdf'))
    plt.show()

    ## validation set
    num_cols = X_val_linear_encoding.shape[1]
    num_rows = math.ceil(num_cols / 3)
    
    fig, axes = plt.subplots(num_rows, 3, figsize=(10, 3 * num_rows))
    axes = axes.ravel()
    
    with open(str(Path(METADATA_DIR) / 'X_names'), 'r') as f:
        column_labels = f.read().splitlines()
    
    for i in range(num_cols):
        axes[i].hist(X_val_linear_encoding[:, i], bins=30, edgecolor='black')
        axes[i].set_title(f'Validation Set {column_labels[i]}')
        axes[i].set_xlabel(f'{column_labels[i]}')
        axes[i].set_ylabel('Frequency')

    ## remove unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    plt.savefig(str(Path(PLOTS_DIR) / 'validation_set_data_distribution_linear_encoding.pdf'))
    plt.show()
    
    ## test set
    num_cols = X_test_linear_encoding.shape[1]
    num_rows = math.ceil(num_cols / 3)
    
    fig, axes = plt.subplots(num_rows, 3, figsize=(10, 3 * num_rows))
    axes = axes.ravel()
    
    with open(str(Path(METADATA_DIR) / 'X_names'), 'r') as f:
        column_labels = f.read().splitlines()
    
    for i in range(num_cols):
        axes[i].hist(X_test_linear_encoding[:, i], bins=30, edgecolor='black')
        axes[i].set_title(f'Test Set {column_labels[i]}')
        axes[i].set_xlabel(f'{column_labels[i]}')
        axes[i].set_ylabel('Frequency')
    
    ## remove unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    plt.savefig(str(Path(PLOTS_DIR) / 'test_set_data_distribution_linear_encoding.pdf'))
    plt.show()
steps_per_epoch = int(np.ceil(len(X_train) / TRAIN_BATCH_SIZE))
LR_DECAY_STEPS = steps_per_epoch * 20   ## decay every ~20 epochs

## MLP

### Create model

Create a classical multi-layer perceptron for regression. Taking some inspiration from [Deep learning-based I-V Global Parameter Extraction for BSIM-CMG](https://www.sciencedirect.com/science/article/pii/S003811012300179X), Solid-State Electronics, Vol. 209, November 2023.

The above publication predicted parameters for BSIM, which is a physics model for advanced transistors that is complicated and might be a similar complexity to the physics we are trying to target/map with these SC qubit hamiltonian values

Reccomended to download a third party app like "Sleep control Center" or "Amphetamine" to prevent computer from sleeping during the many hour/day long training process

### Create Model by Hand

In [ ]:
## just checkin to make sure everything looks good still, we want float32 because its what i made tf use in the 2nd cell
print(X_train.dtype, X_train.shape)
print(y_train.dtype, y_train.shape)

print("TF:", tf.__version__)
print("OS:", platform.platform())
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("Built with ROCm:", tf.test.is_built_with_rocm())
print("GPUs:", tf.config.list_physical_devices("GPU"))
if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    ## n output neurons for n parameters
    if 'Try Both' not in ENCODING_TYPE:
        ## multilayer perceptron (MLP) with 2 input features. MLP is having neurons that adjust rules based on how accurate they can guess things
        model_shape = f'mlp_{len(X_test[0])}_'
        
        ## define the number on neurons in the inner layer (in parameter file)
        model_shape += '_'.join(str(l) for l in NEURONS_PER_LAYER)
    
        model_shape += f'_{len(y_train[0])}'
    else:
        ## multilayer perceptron (MLP) with 2 input features. MLP is having neurons that adjust rules based on how accurate they can guess things
        model_shape_one_hot_encoding = f'mlp_{len(X_test_one_hot_encoding[0])}_'
        model_shape_linear_encoding = f'mlp_{len(X_test_one_hot_encoding[0])}_'
        
        ## define the number on neurons in the inner layer (in parameter file)
        model_shape_one_hot_encoding += '_'.join(str(l) for l in NEURONS_PER_LAYER)
        model_shape_linear_encoding += '_'.join(str(l) for l in NEURONS_PER_LAYER)
    
        model_shape_one_hot_encoding += f'_{len(y_train_one_hot_encoding[0])}'
        model_shape_linear_encoding += f'_{len(y_train_linear_encoding[0])}'

if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    if 'Try Both' not in ENCODING_TYPE:
        if 'one hot' in ENCODING_TYPE:

            ## multihead MODEL (Option B) separate output heads for regression
            ## and classification so each gets the right loss function.
            ## uses the keras Functional API instead of Sequential.

            inp = Input(shape=(len(X_test[0]),), name='input1')
            x = inp
            
            for i, n in enumerate(NEURONS_PER_LAYER):
                x = Dense(n, name='fc{}'.format(i), kernel_initializer='lecun_uniform',
                          kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
                x = LeakyReLU(negative_slope=0.01, name='leaky_relu{}'.format(i))(x)
                x = Dropout(rate=TRAIN_DROPOUT_RATE, name='dropout{}'.format(i))(x)
            
            ## head 1 continuous design parameters to linear activation, mse loss
            continuous_out = Dense(n_continuous, activation='linear', name='continuous',
                                   kernel_initializer='lecun_uniform')(x)
            
            ## head 2 finger_count classification to softmax activation, crossentropy loss
            ## force float32 for softmax numerical stability under mixed_float16 policy
            fingers_out = Dense(n_fc_classes, activation='softmax', name='fingers',
                                kernel_initializer='lecun_uniform', dtype='float32')(x)
            
            model = Model(inputs=inp, outputs=[continuous_out, fingers_out])
        
        else:
            ## linear encoding same Sequential model as before (single output head)
            model = Sequential()
            model.add(Input(shape=(len(X_test[0]),), name='input1'))
            for i, n in enumerate(NEURONS_PER_LAYER):
                model.add(Dense(n, name='fc{}'.format(i), kernel_initializer='lecun_uniform', kernel_regularizer=tf.keras.regularizers.l2(1e-4)))
                model.add(LeakyReLU(negative_slope=0.01, name='leaky_relu{}'.format(i)))
                model.add(Dropout(rate=TRAIN_DROPOUT_RATE, name='dropout{}'.format(i)))
            model.add(Dense(len(y_train[0]), activation='linear', name='fc_output', kernel_initializer='lecun_uniform'))
    
    else:
        model_one_hot_encoding = Sequential()
        model_one_hot_encoding.add(Input(shape=(len(X_test_one_hot_encoding[0]),), name='input1'))
        for i, n in enumerate(NEURONS_PER_LAYER):
            model_one_hot_encoding.add(Dense(n, name='fc{}'.format(i), kernel_initializer='lecun_uniform', kernel_regularizer=tf.keras.regularizers.l2(1e-4)))
            model_one_hot_encoding.add(LeakyReLU(negative_slope=0.01, name='leaky_relu{}'.format(i)))
            model_one_hot_encoding.add(Dropout(rate=TRAIN_DROPOUT_RATE, name='dropout{}'.format(i)))
        model_one_hot_encoding.add(Dense(len(y_train_one_hot_encoding[0]), activation='linear', name='fc_output', kernel_initializer='lecun_uniform'))
    
        model_linear_encoding = Sequential()
        model_linear_encoding.add(Input(shape=(len(X_test_linear_encoding[0]),), name='input1'))
        for i, n in enumerate(NEURONS_PER_LAYER):
            model_linear_encoding.add(Dense(n, name='fc{}'.format(i), kernel_initializer='lecun_uniform', kernel_regularizer=tf.keras.regularizers.l2(1e-4)))
            model_linear_encoding.add(LeakyReLU(negative_slope=0.01, name='leaky_relu{}'.format(i)))
            model_linear_encoding.add(Dropout(rate=TRAIN_DROPOUT_RATE, name='dropout{}'.format(i)))
        model_linear_encoding.add(Dense(len(y_train_linear_encoding[0]), activation='linear', name='fc_output', kernel_initializer='lecun_uniform'))
if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=LR_INITIAL,  
        decay_steps=LR_DECAY_STEPS,        
        decay_rate=LR_DECAY_RATE,          
        staircase=LR_STAIRCASE             
    )
    
    if 'Try Both' not in ENCODING_TYPE:
        if 'one hot' in ENCODING_TYPE:
            ## multihead each head gets its own appropriate loss function
            ## continuous params mse (regression)
            ## finger_count categorical crossentropy (classification)
            ## loss_weights lets you tune the relative importance of each task
            model.compile(
                optimizer=tf.optimizers.Adam(learning_rate=lr_schedule),
                loss={
                    'continuous': TRAIN_LOSS,
                    'fingers':    'categorical_crossentropy',
                },
                loss_weights={
                    'continuous': 1.0,
                    'fingers':    0.5,   ## tune this to balance regression vs classification
                },
                metrics={
                    'continuous': [TRAIN_LOSS],
                    'fingers':    ['accuracy'],
                }
            )
        else:
            model.compile(
                optimizer=tf.optimizers.Adam(learning_rate=lr_schedule),  
                loss=TRAIN_LOSS,                                         
                metrics=[TRAIN_LOSS]                                     
            )
    else:
        model_linear_encoding.compile(
            optimizer=tf.optimizers.Adam(learning_rate=lr_schedule),  
            loss=TRAIN_LOSS,                                         
            metrics=[TRAIN_LOSS]                                     
        )
        model_one_hot_encoding.compile(
            optimizer=tf.optimizers.Adam(learning_rate=lr_schedule),  
            loss=TRAIN_LOSS,                                         
            metrics=[TRAIN_LOSS]                                     
        )
if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
    if 'Try Both' not in ENCODING_TYPE:
        best_model_file = str(Path(MODEL_DIR) / '{}_best_model.keras').format(model_shape)
        last_model_file = str(Path(MODEL_DIR) / '{}_last_model.keras').format(model_shape)
    else:
        best_model_file_one_hot_encoding = str(Path(MODEL_DIR) / '{}_best_model_one_hot_encoding.keras').format(model_shape_one_hot_encoding)
        last_model_file_one_hot_encoding = str(Path(MODEL_DIR) / '{}_last_model_one_hot_encoding.keras').format(model_shape_one_hot_encoding)
    
        best_model_file_linear_encoding = str(Path(MODEL_DIR) / '{}_best_model_linear_encoding.keras').format(model_shape_linear_encoding)
        last_model_file_linear_encoding = str(Path(MODEL_DIR) / '{}_last_model_linear_encoding.keras').format(model_shape_linear_encoding)

Enable training (`train_and_save`) to overwrite the model file.

In [20]:
train_and_save = True

We use Adam optimizer, minimize the Mean Squared Logarithmic Error, and early stop.

#### Training

In [ ]:
## set up monitors and plots for later tracking purposes

if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    class TrainingPlot(tf.keras.callbacks.Callback):
         
        ## this function is called when the training begins
        def on_train_begin(self, logs={}):
            ## initialize the lists for holding the logs, losses
            self.losses = []
            self.val_losses = []
            self.logs = []
        
        ## this function is called at the end of each epoch
        def on_epoch_end(self, epoch, logs={}):
            
            ## append the logs, losses to the lists
            self.logs.append(logs)
            self.losses.append(logs.get('loss'))
            self.val_losses.append(logs.get('val_loss'))
            
            ## before plotting ensure at least 2 epochs have passed
            if len(self.losses) > 1:
                
                ## clear the previous plot
                clear_output(wait=True)
                N = np.arange(0, len(self.losses))
                
                ## plot train loss, train acc, val loss and val acc against epochs passed
                plt.figure()
                plt.plot(N, self.losses, label = "train_loss")
                plt.plot(N, self.val_losses, label = "val_loss")
                plt.title("Training Loss [Epoch {}]".format(epoch))
                plt.xlabel("Epoch #")
                plt.ylabel("Loss/Accuracy")
                plt.legend()
                plt.show()
           
class LearningRateMonitor(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.learning_rates = []

    ## we have to do some checking for versions here or else we will get an Adam error when using this monitor
    def _current_lr(self, optimizer):
        ## look and see if you get "lr" ir "learning_rate" depending on the version
        lr = getattr(optimizer, "lr", None) or getattr(optimizer, "learning_rate", None)

        ## if this is a shecdule then evaluate it at current iteration step
        if isinstance(lr, tf.keras.optimizers.schedules.LearningRateSchedule):
            return float(lr(optimizer.iterations).numpy())

        ## if not a schedule, its a scalar/variable/tensor
        return float(tf.keras.backend.get_value(lr))

    def on_epoch_end(self, epoch, logs=None):
        try:
            lr_val = self._current_lr(self.model.optimizer)
        except Exception:
            ## for anything else fallback
            lr_val = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
        self.learning_rates.append(lr_val)

%%time

## train the model
history = None  
if not KERAS_TUNER and not SWEEP_PARAM_NUM and not SWEEP_DATA_AMOUNT:
    if train_and_save: 
        ## set up early stopping to prevent overfitting by halting training when validation loss stops improving
        early_stopping = EarlyStopping(
            monitor='val_loss',                      ## monitor validation loss for stopping criteria
            mode='min',                              ## stop when the monitored quantity has stopped decreasing
            patience=TRAIN_EARLY_STOPPING_PATIENCE,  ## number of epochs to wait after last improvement
            verbose=1                                ## enable logging when early stopping happens
        )
    
        ## train the model on the training data and validate on a portion of it
        if 'Try Both' not in ENCODING_TYPE:
            plot_callback = TrainingPlot()      ## plot training progress
            lr_monitor = LearningRateMonitor()  ## watch learning rate changes
            
            ## set up model checkpointing to save the model at its best validation loss
            model_checkpoint = ModelCheckpoint(
                filepath=best_model_file,          
                monitor='val_loss',            ## save the model based on validation loss improvement
                mode='min',                    ## favor lower validation loss values for saving (minimize)
                save_best_only=True,           ## save only when validation loss improves
                verbose=0                      ## no logging for model saving
            )

            if 'one hot' in ENCODING_TYPE:
                ## multihead pass targets as a dict keyed by output layer name
                y_train_dict = {'continuous': np.asarray(y_train_cont), 'fingers': np.asarray(y_train_fc)}
                y_val_dict   = {'continuous': np.asarray(y_val_cont),   'fingers': np.asarray(y_val_fc)}
                
                history = model.fit(
                    np.asarray(X_train),  
                    y_train_dict,
                    epochs=400,                   
                    batch_size=TRAIN_BATCH_SIZE,  
                    validation_data=(np.asarray(X_val), y_val_dict),
                    callbacks=[early_stopping, model_checkpoint, plot_callback, lr_monitor],  
                    verbose=1
                )
            else:
                history = model.fit(
                    np.asarray(X_train),  
                    np.asarray(y_train),      
                    epochs=400,                   
                    batch_size=TRAIN_BATCH_SIZE,  
                    validation_data=(np.asarray(X_val), np.asarray(y_val)),  
                    callbacks=[early_stopping, model_checkpoint, plot_callback, lr_monitor],  
                    verbose=1
                )
            
            model.save(last_model_file)  ## save the final model when done training!
        
        else:
            plot_callback_linear_encoding = TrainingPlot()      ## plot training progress
            lr_monitor_linear_encoding = LearningRateMonitor()  ## watch learning rate changes
            
            model_checkpoint_linear_encoding = ModelCheckpoint(
                filepath=best_model_file_linear_encoding,          
                monitor='val_loss',
                mode='min',
                save_best_only=True,
                verbose=0
            )
            
            history_linear_encoding = model_linear_encoding.fit(
                np.asarray(X_train_linear_encoding),  
                np.asarray(y_train_linear_encoding),      
                epochs=400,                   
                batch_size=TRAIN_BATCH_SIZE,  
                validation_data=(np.asarray(X_val_linear_encoding), np.asarray(y_val_linear_encoding)), 
                callbacks=[early_stopping, model_checkpoint_linear_encoding, plot_callback_linear_encoding, lr_monitor_linear_encoding],  
                verbose=1
            )
            
            model_linear_encoding.save(last_model_file_linear_encoding)

            ## one hot
            plot_callback_one_hot_encoding = TrainingPlot()
            lr_monitor_one_hot_encoding = LearningRateMonitor()
            
            model_checkpoint_one_hot_encoding = ModelCheckpoint(
                filepath=best_model_file_one_hot_encoding,          
                monitor='val_loss',
                mode='min',
                save_best_only=True,
                verbose=0
            )
            
            history_one_hot_encoding = model_one_hot_encoding.fit(
                np.asarray(X_train_one_hot_encoding),  
                np.asarray(y_train_one_hot_encoding),      
                epochs=400,                   
                batch_size=TRAIN_BATCH_SIZE,  
                validation_data=(np.asarray(X_val_one_hot_encoding), np.asarray(y_val_one_hot_encoding)), 
                callbacks=[early_stopping, model_checkpoint_one_hot_encoding, plot_callback_one_hot_encoding, lr_monitor_one_hot_encoding],  
                verbose=1
            )
            
            model_one_hot_encoding.save(last_model_file_one_hot_encoding)

Load the saved best model and use it from now on.

In [23]:
if not KERAS_TUNER and not SWEEP_PARAM_NUM and not SWEEP_DATA_AMOUNT:
    if 'Try Both' not in ENCODING_TYPE:
        model = load_model(best_model_file, custom_objects={})
    else:
        model_one_hot_encoding = load_model(best_model_file_one_hot_encoding, custom_objects={})
        model_linear_encoding = load_model(best_model_file_linear_encoding, custom_objects={})

### Sweep total number of parameters to find the right range

In [ ]:
if SWEEP_PARAM_NUM:
    def build_mlp(neurons_per_layer, input_dim, output_dim):
        model = Sequential()
        model.add(Input(shape=(input_dim,), name="input1"))
    
        for i, n in enumerate(neurons_per_layer):
            model.add(Dense(
                n,
                name=f"fc{i}",
                kernel_initializer="lecun_uniform",
                kernel_regularizer=tf.keras.regularizers.l2(1e-4),
            ))
            model.add(LeakyReLU(negative_slope=0.01, name=f"leaky_relu{i}"))
            model.add(Dropout(rate=TRAIN_DROPOUT_RATE, name=f"dropout{i}"))
    
        model.add(Dense(
            output_dim,
            activation="linear",
            name="fc_output",
            kernel_initializer="lecun_uniform",
        ))
        return model
    
    def make_optimizer():
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=LR_INITIAL,
            decay_steps=LR_DECAY_STEPS,
            decay_rate=LR_DECAY_RATE,
            staircase=LR_STAIRCASE
        )
        return tf.optimizers.Adam(learning_rate=lr_schedule)
    
    def train_one_config(neurons_per_layer, seed=0):
        tf.keras.backend.clear_session()
        tf.random.set_seed(seed)
        np.random.seed(seed)
    
        model = build_mlp(
            neurons_per_layer=neurons_per_layer,
            input_dim=X_train.shape[1],
            output_dim=y_train.shape[1],
        )
    
        model.compile(
            optimizer=make_optimizer(),
            loss=TRAIN_LOSS,
            metrics=[TRAIN_LOSS],
        )
    
        early_stopping = EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=TRAIN_EARLY_STOPPING_PATIENCE,
            verbose=0,
            restore_best_weights=True,   ## helps make "best" consistent
        )
    
        history = model.fit(
            np.asarray(X_train),
            np.asarray(y_train),
            validation_data=(np.asarray(X_val), np.asarray(y_val)),
            epochs=400,
            batch_size=TRAIN_BATCH_SIZE,
            callbacks=[early_stopping],
            verbose=0,
        )
    
        best_val_loss = float(np.min(history.history["val_loss"]))
        best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    
        return {
            "neurons_per_layer": str(list(neurons_per_layer)),
            "total_params": int(model.count_params()),
            "best_val_loss": best_val_loss,
            "best_epoch": best_epoch,
        }

    ## here im chosing these depths and widths to explore, and will keep expanding higher until val_loss gets worse ########
    configs = []
    for depth in [3]:
        for width in [64, 256, 1024,1800,500,3500]:## [4, 8, 16, 32, 64, 128, 256, 512, 1024, 1500,1800,2000,2500,3000,3500]
            configs.append([width] * depth)
    
    ## remove duplicates if any
    configs = [list(x) for x in {tuple(c) for c in configs}]
    
    results = []
    for cfg in sorted(configs, key=lambda c: (len(c), c[0])):
        out = train_one_config(cfg, seed=0)
        results.append(out)
        print(out)
    
    df = pd.DataFrame(results).sort_values("total_params")
if SWEEP_PARAM_NUM:
    ## save the data

    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = "sweep_outputs"
    os.makedirs(out_dir, exist_ok=True)
    
    csv_path = os.path.join(out_dir, f"sweep_results_{run_id}.csv")
    df.to_csv(csv_path, index=False)
    print("Saved:", csv_path)
if SWEEP_PARAM_NUM:
    plt.figure()
    plt.scatter(df["total_params"], df["best_val_loss"])
    plt.xscale("log")  ## will be helpful if params grow fast
    plt.xlabel("Total parameters (log scale)")
    plt.ylabel("Best val_loss")
    plt.title("Best val_loss vs model size")
    plt.savefig(str(Path(PLOTS_DIR) / 'params_vs_loss.png'))
    plt.show()
if SWEEP_PARAM_NUM:
    ## extract depth/width from the stored string like "[32, 32, 32]"
    df2 = df.copy()
    df2["depth"] = df2["neurons_per_layer"].apply(lambda s: len(eval(s)))
    df2["width"] = df2["neurons_per_layer"].apply(lambda s: eval(s)[0])
    
    plt.figure()
    sc = plt.scatter(
        df2["total_params"],
        df2["best_val_loss"],
        c=df2["depth"],
        s=20 + 10*np.log2(df2["width"]),
    )
    plt.xscale("log")
    plt.xlabel("Total parameters (log scale)")
    plt.ylabel("Best val_loss")
    plt.title("Best val_loss vs model size (color=depth, size=width)")
    plt.colorbar(sc, label="Depth")
    plt.savefig(str(Path(PLOTS_DIR) / 'params_vs_loss_width_color_coded.png'))
    plt.show()
if SWEEP_PARAM_NUM:
    df.to_csv(str(Path(RESULTS_DIR) / "validation/sweep_results.csv"), index=False)

    old = pd.read_csv(str(Path(RESULTS_DIR) / "validation/sweep_results.csv"))
    combined = pd.concat([old, df], ignore_index=True).drop_duplicates(
        subset=["neurons_per_layer"], keep="last"
    )
    combined.to_csv(str(Path(RESULTS_DIR) / "validation/sweep_results.csv"), index=False)

### Sweep amount of data used in training to determine if data amount is limiting

In [ ]:
if SWEEP_DATA_AMOUNT:

    FIXED_DEPTH = 1          
    FIXED_WIDTH = 1024        
    FIXED_NEURONS = [FIXED_WIDTH] * FIXED_DEPTH

    ## this is the fraction of the original dataset we want to train on to compare
    TRAIN_FRACTIONS = np.linspace(0.3,1,20)  

    ## averaging over many seeds becasue when you decrease the training data sizes there is more variation due to luck,
    ## this gives better comparison for the impact of just the training data size
    SWEEP_SEEDS = [0, 1, 2, 3, 4]  

    def build_mlp(neurons_per_layer, input_dim, output_dim):
        model = Sequential()
        model.add(Input(shape=(input_dim,), name="input1"))

        for i, n in enumerate(neurons_per_layer):
            model.add(Dense(
                n,
                name=f"fc{i}",
                kernel_initializer="lecun_uniform",
                kernel_regularizer=tf.keras.regularizers.l2(1e-4),
            ))
            model.add(LeakyReLU(negative_slope=0.01, name=f"leaky_relu{i}"))
            model.add(Dropout(rate=TRAIN_DROPOUT_RATE, name=f"dropout{i}"))

        model.add(Dense(
            output_dim,
            activation="linear",
            name="fc_output",
            kernel_initializer="lecun_uniform",
        ))
        return model

    def make_optimizer():
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=LR_INITIAL,
            decay_steps=LR_DECAY_STEPS,
            decay_rate=LR_DECAY_RATE,
            staircase=LR_STAIRCASE
        )
        return tf.optimizers.Adam(learning_rate=lr_schedule)

    def make_subset(X, y, frac, seed):
        """Return a random subset (without replacement) of X/y of size frac."""
        assert 0 < frac <= 1.0
        n = len(X)
        m = max(1, int(np.floor(frac * n)))
        rng = np.random.default_rng(seed)
        idx = rng.choice(n, size=m, replace=False)
        return X[idx], y[idx], m

    def train_one_fraction(neurons_per_layer, train_frac, seed=0):
        tf.keras.backend.clear_session()
        tf.random.set_seed(seed)
        np.random.seed(seed)

        ## subset only the TRAIN split
        X_sub, y_sub, n_sub = make_subset(np.asarray(X_train), np.asarray(y_train), train_frac, seed)

        model = build_mlp(
            neurons_per_layer=neurons_per_layer,
            input_dim=X_train.shape[1],
            output_dim=y_train.shape[1],
        )

        model.compile(
            optimizer=make_optimizer(),
            loss=TRAIN_LOSS,
            metrics=[TRAIN_LOSS],
        )

        early_stopping = EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=TRAIN_EARLY_STOPPING_PATIENCE,
            verbose=0,
            restore_best_weights=True,
        )

        history = model.fit(
            X_sub,
            y_sub,
            validation_data=(np.asarray(X_val), np.asarray(y_val)),
            epochs=400,
            batch_size=TRAIN_BATCH_SIZE,
            callbacks=[early_stopping],
            verbose=0,
        )

        best_val_loss = float(np.min(history.history["val_loss"]))
        best_epoch = int(np.argmin(history.history["val_loss"]) + 1)

        return {
            "train_frac": float(train_frac),
            "train_n": int(n_sub),
            "seed": int(seed),
            "neurons_per_layer": str(list(neurons_per_layer)),
            "total_params": int(model.count_params()),
            "best_val_loss": best_val_loss,
            "best_epoch": best_epoch,
        }

    ## do the sweep ######
    results = []
    for frac in TRAIN_FRACTIONS:
        for seed in SWEEP_SEEDS:
            out = train_one_fraction(FIXED_NEURONS, train_frac=frac, seed=seed)
            results.append(out)
            print(out)

    df = pd.DataFrame(results).sort_values(["train_frac", "seed"]).reset_index(drop=True)

    ## do the mean/std over the many seeds to average
    summary = (
        df.groupby(["train_frac", "train_n", "total_params"], as_index=False)
          .agg(best_val_loss_mean=("best_val_loss", "mean"),
               best_val_loss_std=("best_val_loss", "std"),
               best_epoch_mean=("best_epoch", "mean"))
          .sort_values("train_frac")
          .reset_index(drop=True)
    )

    ## save the data ####
    run_id = time.strftime("%Y%m%d_%H%M%S")
    out_dir = os.path.join("sweeps", f"data_fraction_sweep_{run_id}")
    os.makedirs(out_dir, exist_ok=True)

    df_path = os.path.join(out_dir, "sweep_raw.csv")
    summary_path = os.path.join(out_dir, "sweep_summary.csv")
    meta_path = os.path.join(out_dir, "metadata.json")
    fig_path = os.path.join(out_dir, "val_loss_vs_train_fraction.png")

    df.to_csv(df_path, index=False)
    summary.to_csv(summary_path, index=False)

    metadata = {
        "run_id": run_id,
        "fixed_depth": FIXED_DEPTH,
        "fixed_width": FIXED_WIDTH,
        "fixed_neurons": FIXED_NEURONS,
        "train_fractions": TRAIN_FRACTIONS,
        "seeds": SWEEP_SEEDS,
        "train_loss": str(TRAIN_LOSS),
        "batch_size": int(TRAIN_BATCH_SIZE),
        "early_stopping_patience": int(TRAIN_EARLY_STOPPING_PATIENCE),
        "notes": "Subset sampling is applied only to X_train/y_train; validation is fixed.",
    }
if SWEEP_DATA_AMOUNT:
    def _jsonify(o):
        import numpy as np
        ## numpy array > list
        if isinstance(o, np.ndarray):
            return o.tolist()
        ## numpy scalars > python scalars
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        ## fallback let json try, otherwise stringify
        return o

    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2, default=_jsonify)

    plt.figure()
    if len(SWEEP_SEEDS) > 1:
        yerr = summary["best_val_loss_std"].fillna(0.0).to_numpy()
        plt.errorbar(summary["train_frac"], summary["best_val_loss_mean"], yerr=yerr, marker="o")
        plt.ylabel("Best val loss (mean +/- std)")
    else:
        plt.plot(summary["train_frac"], summary["best_val_loss_mean"], marker="o")
        plt.ylabel("Best val loss")

    plt.xlabel("Training data fraction")
    plt.title(f"Val loss vs training fraction (depth={FIXED_DEPTH}, width={FIXED_WIDTH})")
    plt.tight_layout()
    plt.savefig(fig_path, dpi=200)
    plt.show()

    print(f"\nSaved:\n- {df_path}\n- {summary_path}\n- {meta_path}\n- {fig_path}")

### Keras Tuner to Find Best Hyperparameters and Model

Run this if you want to use keras tuner to make the model rather than doing it by hand

In [ ]:

if KERAS_TUNER and not SWEEP_PARAM_NUM:
    if 'Try Both' not in ENCODING_TYPE:
        def build_hypermodel(hp):
            tf.keras.backend.clear_session()
            gc.collect()
            
            ## iMPROVEMENT 1 Search over variable depth (15)
            ## with only ~300 training samples, fewer layers often wins.
            n_layers = hp.Int('n_layers', min_value=1, max_value=5, default=3)
            neurons_per_layer = [hp.Int(f'neurons_{i}', min_value=32, max_value=512, step=32) for i in range(n_layers)]
            
            ## iMPROVEMENT 2 Wider regularization search
            dropout_rate = hp.Float('dropout_rate', 0.0, 0.4, step=0.05)
            l2_reg = hp.Float('l2_reg', 1e-5, 1e-2, sampling='LOG', default=1e-4)

            ## iMPROVEMENT 3 Tunable learning rate with wider range
            lr_initial = hp.Float('learning_rate', 1e-4, 3e-3, sampling='LOG', default=5e-4)
            ## lR schedule removed ReduceLROnPlateau callback handles decay instead
            ## iMPROVEMENT 4 Optional Batch Normalization
            use_batchnorm = hp.Boolean('use_batchnorm', default=True)
            
            if 'one hot' in ENCODING_TYPE:
                ## iMPROVEMENT 5 Wider fc_loss_weight range
                ## higher weight = more emphasis on getting finger_count right
                fc_loss_weight = hp.Float('fc_loss_weight', 0.1, 10.0, sampling='LOG', default=1.0)
                
                inp = Input(shape=(len(X_test[0]),), name='input1')
                x = inp
                for i, n_units in enumerate(neurons_per_layer):
                    x = Dense(n_units, name=f'fc{i}', kernel_initializer='he_normal',
                              kernel_regularizer=tf.keras.regularizers.l2(l2_reg))(x)
                    if use_batchnorm:
                        x = tf.keras.layers.BatchNormalization(name=f'bn{i}')(x)
                    x = LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}')(x)
                    x = Dropout(rate=dropout_rate, name=f'dropout{i}')(x)
                
                continuous_out = Dense(n_continuous, activation='linear', name='continuous',
                                       kernel_initializer='he_normal')(x)
                fingers_out = Dense(n_fc_classes, activation='softmax', name='fingers',
                                    kernel_initializer='he_normal', dtype='float32')(x)
                
                model = Model(inputs=inp, outputs=[continuous_out, fingers_out])
                model.compile(
                    optimizer=tf.optimizers.Adam(learning_rate=lr_initial),
                    loss={'continuous': TRAIN_LOSS, 'fingers': 'categorical_crossentropy'},
                    loss_weights={'continuous': 1.0, 'fingers': fc_loss_weight},
                    metrics={'continuous': [TRAIN_LOSS], 'fingers': ['accuracy']}
                )
            else:
                model = Sequential()
                model.add(Input(shape=(len(X_test[0]),), name='input1'))
                for i, n_units in enumerate(neurons_per_layer):
                    model.add(Dense(n_units, name=f'fc{i}', kernel_initializer='he_normal',
                                    kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
                    if use_batchnorm:
                        model.add(tf.keras.layers.BatchNormalization(name=f'bn{i}'))
                    model.add(LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}'))
                    model.add(Dropout(rate=dropout_rate, name=f'dropout{i}'))
                model.add(Dense(len(y_train[0]), name='output', kernel_initializer='he_normal'))
                model.compile(optimizer=tf.optimizers.Adam(learning_rate=lr_initial), 
                              loss=TRAIN_LOSS, metrics=[TRAIN_LOSS])
            return model
    else:
        def build_hypermodel_one_hot_encoding(hp):
            tf.keras.backend.clear_session()
            gc.collect()
            n_layers = hp.Int('n_layers', min_value=1, max_value=5, default=3)
            neurons_per_layer = [hp.Int(f'neurons_{i}', min_value=32, max_value=512, step=32) for i in range(n_layers)]
            dropout_rate = hp.Float('dropout_rate', 0.0, 0.4, step=0.05)
            l2_reg = hp.Float('l2_reg', 1e-5, 1e-2, sampling='LOG', default=1e-4)
            use_batchnorm = hp.Boolean('use_batchnorm', default=True)
            model_one_hot_encoding = Sequential()
            model_one_hot_encoding.add(Input(shape=(len(X_test_one_hot_encoding[0]),), name='input1'))
            for i, n in enumerate(neurons_per_layer):
                model_one_hot_encoding.add(Dense(n, name=f'fc{i}', kernel_initializer='he_normal', kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
                if use_batchnorm:
                    model_one_hot_encoding.add(tf.keras.layers.BatchNormalization(name=f'bn{i}'))
                model_one_hot_encoding.add(LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}'))
                model_one_hot_encoding.add(Dropout(rate=dropout_rate, name=f'dropout{i}'))
            model_one_hot_encoding.add(Dense(len(y_train_one_hot_encoding[0]), name='output', kernel_initializer='he_normal'))
            lr_initial = hp.Float('learning_rate', 1e-4, 3e-3, sampling='LOG', default=5e-4)
            ## lR schedule removed ReduceLROnPlateau callback handles decay instead
            model_one_hot_encoding.compile(optimizer=tf.optimizers.Adam(learning_rate=lr_initial), 
                          loss=TRAIN_LOSS, metrics=[TRAIN_LOSS])
            return model_one_hot_encoding

        def build_hypermodel_linear_encoding(hp):
            tf.keras.backend.clear_session()
            gc.collect()
            n_layers = hp.Int('n_layers', min_value=1, max_value=5, default=3)
            neurons_per_layer = [hp.Int(f'neurons_{i}', min_value=32, max_value=512, step=32) for i in range(n_layers)]
            dropout_rate = hp.Float('dropout_rate', 0.0, 0.4, step=0.05)
            l2_reg = hp.Float('l2_reg', 1e-5, 1e-2, sampling='LOG', default=1e-4)
            use_batchnorm = hp.Boolean('use_batchnorm', default=True)
            model_linear_encoding = Sequential()
            model_linear_encoding.add(Input(shape=(len(X_test_linear_encoding[0]),), name='input1'))
            for i, n in enumerate(neurons_per_layer):
                model_linear_encoding.add(Dense(n, name=f'fc{i}', kernel_initializer='he_normal', kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
                if use_batchnorm:
                    model_linear_encoding.add(tf.keras.layers.BatchNormalization(name=f'bn{i}'))
                model_linear_encoding.add(LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}'))
                model_linear_encoding.add(Dropout(rate=dropout_rate, name=f'dropout{i}'))
            model_linear_encoding.add(Dense(len(y_train_linear_encoding[0]), name='output', kernel_initializer='he_normal'))
            lr_initial = hp.Float('learning_rate', 1e-4, 3e-3, sampling='LOG', default=5e-4)
            ## lR schedule removed ReduceLROnPlateau callback handles decay instead
            model_linear_encoding.compile(optimizer=tf.optimizers.Adam(learning_rate=lr_initial), 
                          loss=TRAIN_LOSS, metrics=[TRAIN_LOSS])
            return model_linear_encoding
## iMPROVEMENT 6 Use BayesianOptimization instead of RandomSearch
## bayesian optimization is much more sampleefficient at finding good hyperparams.

if 'Try Both' not in ENCODING_TYPE:
    if KERAS_TUNER and not SWEEP_PARAM_NUM:
        tuner = BayesianOptimization(
            build_hypermodel,
            objective='val_loss',
            max_trials=KERAS_TUNER_TRIALS,
            executions_per_trial=2,  ## improvement 7 average over 2 runs per trial
            directory=KERAS_DIR + '/hyper_tuning_v3',
            project_name='mlp_tuning_bayesian',
        )
else:
    if KERAS_TUNER and not SWEEP_PARAM_NUM:
        tuner_linear_encoding = BayesianOptimization(
            build_hypermodel_linear_encoding,
            objective='val_loss',
            max_trials=KERAS_TUNER_TRIALS,
            executions_per_trial=2,
            directory=KERAS_DIR + '/hyper_tuning_linear_encoding_v2',
            project_name='mlp_tuning_linear_bayesian'
        )

        tuner_one_hot_encoding = BayesianOptimization(
            build_hypermodel_one_hot_encoding,
            objective='val_loss',
            max_trials=KERAS_TUNER_TRIALS,
            executions_per_trial=2,
            directory=KERAS_DIR + '/hyper_tuning_one_hot_encoding_v2',
            project_name='mlp_tuning_onehot_bayesian'
        )

if KERAS_TUNER and not SWEEP_PARAM_NUM:
    ## setup Callbacks
    early_stopping = EarlyStopping(
        monitor='val_loss',
        mode='min',
        patience=TRAIN_EARLY_STOPPING_PATIENCE,
        verbose=1
    )
    
    ## iMPROVEMENT 8 Add ReduceLROnPlateau
    ## if val_loss plateaus, reduce LR to escape local minima
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=max(10, TRAIN_EARLY_STOPPING_PATIENCE // 3),
        min_lr=1e-6,
        verbose=1
    )
if KERAS_TUNER and not SWEEP_PARAM_NUM:
    if 'Try Both' not in ENCODING_TYPE:
        if 'one hot' in ENCODING_TYPE:
            ## multihead pass targets as dict
            tuner.search(np.asarray(X_train), 
                         {'continuous': np.asarray(y_train_cont), 'fingers': np.asarray(y_train_fc)},
                         epochs=400, 
                         batch_size=TRAIN_BATCH_SIZE, 
                         validation_data=(np.asarray(X_val), {'continuous': np.asarray(y_val_cont), 'fingers': np.asarray(y_val_fc)}),
                         callbacks=[early_stopping, reduce_lr], 
                         verbose=1)
        else:
            tuner.search(np.asarray(X_train), 
                         np.asarray(y_train), 
                         epochs=400, 
                         batch_size=TRAIN_BATCH_SIZE, 
                         validation_data=(np.asarray(X_val), np.asarray(y_val)),
                         callbacks=[early_stopping, reduce_lr], 
                         verbose=1)
    else:
        tuner_one_hot_encoding.search(np.asarray(X_train_one_hot_encoding), 
                     np.asarray(y_train_one_hot_encoding), 
                     epochs=400, 
                     batch_size=TRAIN_BATCH_SIZE, 
                     validation_data=(np.asarray(X_val_one_hot_encoding), np.asarray(y_val_one_hot_encoding)),
                     callbacks=[early_stopping, reduce_lr], 
                     verbose=1)

        tuner_linear_encoding.search(np.asarray(X_train_linear_encoding), 
                     np.asarray(y_train_linear_encoding), 
                     epochs=400, 
                     batch_size=TRAIN_BATCH_SIZE, 
                     validation_data=(np.asarray(X_val_linear_encoding), np.asarray(y_val_linear_encoding)),
                     callbacks=[early_stopping, reduce_lr], 
                     verbose=1)

encoding = ENCODING_TYPE.replace(' ','_')
if KERAS_TUNER and not SWEEP_PARAM_NUM:
    if 'Try Both' not in ENCODING_TYPE:
        os.makedirs(MODEL_DIR, exist_ok=True)
        best_model_file= f'{MODEL_DIR}/best_keras_model_{encoding}_encoding.keras'

        best_model = tuner.get_best_models(1)[0]
        best_model.save(best_model_file)

        ## gpu is thowing errors, lets try to clear its memory
        tf.keras.backend.clear_session()
        gc.collect()
        
        ## lets not compile to help with the memory bug
        with tf.device('/CPU:0'):
            loaded_model = load_model(best_model_file, compile=False)
    else:
        os.makedirs(MODEL_DIR, exist_ok=True)
        best_model_file_linear = str(Path(MODEL_DIR) / 'best_keras_model_linear_encoding.keras')
        best_model_file_onehot = str(Path(MODEL_DIR) / 'best_keras_model_one_hot_encoding.keras')
        
        best_linear_model = tuner_linear_encoding.get_best_models(1)[0]
        best_onehot_model = tuner_one_hot_encoding.get_best_models(1)[0]
        
        best_linear_model.save(best_model_file_linear)
        best_onehot_model.save(best_model_file_onehot)
        
        ## gpu is thowing errors, lets try to clear its memory
        tf.keras.backend.clear_session()
        gc.collect()
        
        ## lets not compile to help with the memory bug
        with tf.device('/CPU:0'):
            loaded_linear_model = load_model(best_model_file_linear, compile=False)
            loaded_onehot_model = load_model(best_model_file_onehot, compile=False)

### View the model

In [38]:
if KERAS_TUNER and not SWEEP_PARAM_NUM:
    if 'Try Both' not in ENCODING_TYPE:
        best_model.summary()
    else:
        best_onehot_model.summary()
        best_linear_model.summary()
        
if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    if 'Try Both' not in ENCODING_TYPE:
        print("\n---- Model Summary ----")
        model.summary()
    else:
        print("\n---- Linear Encoding Model Summary ----")
        model_linear_encoding.summary()
        
        print("\n---- One-Hot Encoding Model Summary ----")
        model_one_hot_encoding.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input1 (InputLayer) │ (None, 6)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc0 (Dense)         │ (None, 256)       │      1,792 │ input1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_relu0         │ (None, 256)       │          0 │ fc0[0][0]         │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout0 (Dropout)  │ (None, 256)       │          0 │ leaky_relu0[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc1 (Dense)         │ (None, 256)       │     65,792 │ dropout0[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_relu1         │ (None, 256)       │          0 │ fc1[0][0]         │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout1 (Dropout)  │ (None, 256)       │          0 │ leaky_relu1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc2 (Dense)         │ (None, 448)       │    115,136 │ dropout1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_relu2         │ (None, 448)       │          0 │ fc2[0][0]         │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout2 (Dropout)  │ (None, 448)       │          0 │ leaky_relu2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ continuous (Dense)  │ (None, 3)         │      1,347 │ dropout2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fingers (Dense)     │ (None, 10)        │      4,490 │ dropout2[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 188,557 (736.55 KB)

 Trainable params: 188,557 (736.55 KB)

 Non-trainable params: 0 (0.00 B)

### Evaluation

Although we may plot and print many metrics, we focus only on **Mean Squared Error (MSE).**

Plot training history.

In [ ]:
## run %matplotlib ipympl here if you want
%matplotlib inline

### Visualize gradients for best model

In [ ]:
if 'Try Both' not in ENCODING_TYPE:
    if KERAS_TUNER and not SWEEP_PARAM_NUM and VISUALIZE_GRADIENTS:
        class GradientNormLogger(tf.keras.callbacks.Callback):
            """
            Logs gradient norms on a fixed probe batch at epoch end.
            Useful to detect exploding (huge/NaN/Inf) or vanishing (near-zero) gradients.
            """
            def __init__(
                self,
                x_probe,
                y_probe,
                layer_name_prefixes=("fc", "output"),
                log_every=1,
                max_items_per_prefix=10,
                verbose=1
            ):
                super().__init__()
                self.x_probe = tf.convert_to_tensor(x_probe)
                self.y_probe = tf.convert_to_tensor(y_probe)
                self.layer_name_prefixes = tuple(layer_name_prefixes)
                self.log_every = int(log_every)
                self.max_items_per_prefix = int(max_items_per_prefix)
                self.verbose = int(verbose)
        
                ## will store a list of dicts, one per logged epoch
                self.records = []
        
            def _want_layer(self, var_name: str) -> bool:
                ## var_name looks like "fc0/kernel0", "output/bias0", etc.
                return any(var_name.startswith(pfx) for pfx in self.layer_name_prefixes)
        
            def on_epoch_end(self, epoch, logs=None):
                logs = logs or {}
                if (epoch + 1) % self.log_every != 0:
                    return
        
                ## compute grads on the probe batch
                with tf.GradientTape() as tape:
                    y_pred = self.model(self.x_probe, training=True)
                    ## use the compiled loss exactly as training does
                    loss = self.model.compiled_loss(self.y_probe, y_pred)
        
                grads = tape.gradient(loss, self.model.trainable_weights)
        
                rec = {"epoch": int(epoch + 1), "probe_loss": float(loss.numpy())}
        
                ## aggregate pervariable gradient norms and also perlayer summary
                per_layer = {}  ## layer_name > list of norms
        
                for w, g in zip(self.model.trainable_weights, grads):
                    if g is None:
                        continue
        
                    wname = w.name.split(":")[0]  ## remove "0"
                    if not self._want_layer(wname):
                        continue
        
                    ## l2 norm of gradient tensor
                    g_norm = tf.linalg.global_norm([g]).numpy().item()
                    rec[f"grad_norm__{wname}"] = float(g_norm)

                    ## perlayer group "fc0/kernel" > "fc0", "output/bias" > "output"
                    layer_key = wname.split("/")[0]
                    per_layer.setdefault(layer_key, []).append(g_norm)
        
                ## summaries per layer (mean / max across kernel+bias etc.)
                for layer_key, norms in per_layer.items():
                    rec[f"grad_mean__{layer_key}"] = float(np.mean(norms))
                    rec[f"grad_max__{layer_key}"] = float(np.max(norms))
        
                ## track global grad norm too (all trainable weights)
                g_all = [g for g in grads if g is not None]
                if g_all:
                    rec["grad_global_norm"] = float(tf.linalg.global_norm(g_all).numpy().item())
                else:
                    rec["grad_global_norm"] = float("nan")
        
                self.records.append(rec)
        
                ## also push a few values into logs so they show up in History
                ## (keras History only stores scalars)
                for k in list(rec.keys()):
                    if k not in ("epoch",):
                        logs[k] = rec[k]
        
                ## optional console print (keep it short)
                if self.verbose:
                    ## show fc0 and output summaries if present
                    msg_parts = [f"[Grad] epoch={rec['epoch']} probe_loss={rec['probe_loss']:.6g} global={rec['grad_global_norm']:.3g}"]
                    for lk in ("fc0", "output"):
                        if f"grad_mean__{lk}" in rec:
                            msg_parts.append(f"{lk}:mean={rec[f'grad_mean__{lk}']:.3g} max={rec[f'grad_max__{lk}']:.3g}")
                    print(" | ".join(msg_parts))
        
            def to_csv(self, path: str):
                import csv
                if not self.records:
                    return
                ## collect all keys
                keys = sorted({k for r in self.records for k in r.keys()})
                with open(path, "w", newline="") as f:
                    w = csv.DictWriter(f, fieldnames=keys)
                    w.writeheader()
                    w.writerows(self.records)

if 'Try Both' not in ENCODING_TYPE:
    if KERAS_TUNER and not SWEEP_PARAM_NUM and VISUALIZE_GRADIENTS:
        ## choose a fixed probe batch (small to avoid overhead)
        probe_n = min(256, len(X_train))
        x_probe = np.asarray(X_train[:probe_n])
        y_probe = np.asarray(y_train[:probe_n])
        
        grad_logger = GradientNormLogger(
            x_probe=x_probe,
            y_probe=y_probe,
            layer_name_prefixes=("fc0", "output"),  ## just your one hidden layer + output
            log_every=1,
            verbose=1
        )
        ## now both searches should have completed, so lets get the best hyperparams
        ## and retrain with history saved so we can look at it
        best_hp  = tuner.get_best_hyperparameters(1)[0]
        
        ## make the models again using the best hyperparams
        model    = tuner.hypermodel.build(best_hp)

        lr_monitor = LearningRateMonitor()  ## make learning rate monitor
        
        ## retrain with history so we can plot it
        if 'one hot' in ENCODING_TYPE:
            y_train_dict = {'continuous': np.asarray(y_train_cont), 'fingers': np.asarray(y_train_fc)}
            y_val_dict   = {'continuous': np.asarray(y_val_cont),   'fingers': np.asarray(y_val_fc)}
            history = model.fit(
                np.asarray(X_train), y_train_dict,
                epochs=400, batch_size=TRAIN_BATCH_SIZE,
                validation_data=(np.asarray(X_val), y_val_dict),
                callbacks=[early_stopping, lr_monitor, grad_logger], verbose=1
            )
        else:
            history = model.fit(
                np.asarray(X_train), np.asarray(y_train),
                epochs=400, batch_size=TRAIN_BATCH_SIZE,
                validation_data=(np.asarray(X_val), np.asarray(y_val)),
                callbacks=[early_stopping, lr_monitor, grad_logger], verbose=1
            )
        
        ## save gradient logs
        grad_logger.to_csv(f"{PLOTS_DIR}/{encoding}_gradients.csv")        
        ## keep getting a memory allocation error on EAF so lets free everything after the first
        ## model fit, before moving to the next one
        
        del model
        tf.keras.backend.clear_session()
        gc.collect()
if 'Try Both' not in ENCODING_TYPE:
    if KERAS_TUNER and not SWEEP_PARAM_NUM and VISUALIZE_GRADIENTS:
        dfg = pd.DataFrame(grad_logger.records)
        
        ## plot global grad norm
        plt.figure()
        plt.plot(dfg["epoch"], dfg["grad_global_norm"])
        plt.yscale("log")  ## very helpful to see vanishing/exploding
        plt.title("Gradient tracking for a single batch across epochs")
        plt.xlabel("Epoch")
        plt.ylabel("Gradient magnitude (log scale)")
        plt.tight_layout()
        plt.savefig(f"{PLOTS_DIR}/{encoding}_grad_global_norm.pdf")
        plt.show()
        
        ## plot fc0 and output mean norms if present
        for lk in ["fc0", "output"]:
            col = f"grad_mean__{lk}"
            if col in dfg.columns:
                plt.figure()
                plt.plot(dfg["epoch"], dfg[col])
                plt.yscale("log")
                plt.title(f"Gradient Mean Norm: {lk} (probe batch)")
                plt.xlabel("Epoch")
                plt.ylabel("Mean grad norm (log scale)")
                plt.tight_layout()
                plt.savefig(f"{PLOTS_DIR}/{encoding}_grad_mean_{lk}.pdf")
                plt.show()

### Look at best model

In [ ]:
if 'Try Both' not in ENCODING_TYPE:
    if KERAS_TUNER and not SWEEP_PARAM_NUM and not VISUALIZE_GRADIENTS:
        ## now both searches should have completed, so lets get the best hyperparams
        ## and retrain with history saved so we can look at it
        best_hp  = tuner.get_best_hyperparameters(1)[0]
        
        ## make the models again using the best hyperparams
        model    = tuner.hypermodel.build(best_hp)

        lr_monitor = LearningRateMonitor()  ## make learning rate monitor
        
        ## retrain with history so we can plot it
        if 'one hot' in ENCODING_TYPE:
            y_train_dict = {'continuous': np.asarray(y_train_cont), 'fingers': np.asarray(y_train_fc)}
            y_val_dict   = {'continuous': np.asarray(y_val_cont),   'fingers': np.asarray(y_val_fc)}
            history = model.fit(
                np.asarray(X_train), y_train_dict,
                epochs=400, batch_size=TRAIN_BATCH_SIZE,
                validation_data=(np.asarray(X_val), y_val_dict),
                callbacks=[early_stopping, lr_monitor], verbose=1
            )
        else:
            history = model.fit(
                np.asarray(X_train), np.asarray(y_train),
                epochs=400, batch_size=TRAIN_BATCH_SIZE,
                validation_data=(np.asarray(X_val), np.asarray(y_val)),
                callbacks=[early_stopping, lr_monitor], verbose=1
            )
        
        ## keep getting a memory allocation error on EAF so lets free everything after the first
        ## model fit, before moving to the next one
        
        del model
        tf.keras.backend.clear_session()
        gc.collect()
    
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='best')
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/{save_encoding}_history.pdf')
    plt.show()

    plt.plot(lr_monitor.learning_rates)
    plt.title("Learning Rate over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Learning Rate")
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/{save_encoding}_learning_rate.pdf')
    plt.show()
else:
    if KERAS_TUNER and not SWEEP_PARAM_NUM:
        ## now both searches should have completed, so lets get the best hyperparams
        ## and retrain with history saved so we can look at it
        best_hp_linear  = tuner_linear_encoding.get_best_hyperparameters(1)[0]

        ## make the models again using the best hyperparams
        model_linear    = tuner_linear_encoding.hypermodel.build(best_hp_linear)

        lr_monitor_linear_encoding = LearningRateMonitor()  ## make learning rate monitor
        
        ## retrain with history so we can plot it
        history_linear_encoding = model_linear.fit(
            np.asarray(X_train_linear_encoding),
            np.asarray(y_train_linear_encoding),
            epochs=400,
            batch_size=TRAIN_BATCH_SIZE,
            validation_data=(np.asarray(X_val_linear_encoding), np.asarray(y_val_linear_encoding)),
            callbacks=[early_stopping, lr_monitor_linear_encoding],
            verbose=1
        )
        
        ## keep getting a memory allocation error on EAF so lets free everything after the first
        ## model fit, before moving to the next one
        
        del model_linear
        tf.keras.backend.clear_session()
        gc.collect()

    plt.plot(history_linear_encoding.history['loss'])
    plt.plot(history_linear_encoding.history['val_loss'])
    plt.title('Model loss linear encoding')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='best')
    plt.tight_layout()
    plt.savefig(str(Path(PLOTS_DIR) / 'linear_encoding_history.pdf'))
    plt.show()

    plt.plot(lr_monitor_linear_encoding.learning_rates)
    plt.title("Learning Rate over Epochs linear encoding")
    plt.xlabel("Epoch")
    plt.ylabel("Learning Rate")
    plt.tight_layout()
    plt.savefig(str(Path(PLOTS_DIR) / 'linear_encoding_learning_rate.pdf'))
    plt.show()
## take a look at the loss results and save them

## clear everything so we start with a blank slate memory wise
tf.keras.backend.clear_session()                         ## clear tf backend
gc.collect()                                             ## just collect stray things

def get_loss(eval_out):
    ## eval_out can be float, list/tuple/ndarray, or dict
    if isinstance(eval_out, dict):
        return float(eval_out.get('loss', list(eval_out.values())[0]))
    if isinstance(eval_out, (list, tuple, np.ndarray)):
        return float(eval_out[0])
    return float(eval_out)

def mlp_signature(m, prefix="mlp"):
    ## get the model input sizze first
    in_shape = m.input_shape
    if isinstance(in_shape, list):     ## just get the first one
        in_shape = in_shape[0]
    dims = [d for d in in_shape[1:] if d is not None]
    input_size = int(np.prod(dims)) if dims else "None"

    ## now get the dense laysers in order
    dense_units = [l.units for l in m.layers if isinstance(l, Dense)]

    parts = [prefix, str(input_size)] + [str(u) for u in dense_units]
    return "_".join(parts)

if 'Try Both' not in ENCODING_TYPE:
    model = load_model(best_model_file, compile=False)
    
    if 'one hot' in ENCODING_TYPE:
        ## multihead model compile with separate losses to get meaningful eval
        model.compile(
            optimizer='adam',
            loss={'continuous': TRAIN_LOSS, 'fingers': 'categorical_crossentropy'},
            loss_weights={'continuous': 1.0, 'fingers': 0.5},
            metrics={'continuous': [TRAIN_LOSS], 'fingers': ['accuracy']}
        )
        eval_out = model.evaluate(
            np.asarray(X_test),
            {'continuous': np.asarray(y_test_cont), 'fingers': np.asarray(y_test_fc)})
        test_loss_result = float(eval_out[0])  ## total weighted loss
        print(f"  continuous {TRAIN_LOSS}: {eval_out[1]:.6f}")
        print(f"  fingers crossentropy: {eval_out[2]:.6f}")
        if len(eval_out) > 4:
            print(f"  fingers accuracy: {eval_out[4]:.4f}")
    else:
        model.compile(optimizer='adam', loss=TRAIN_LOSS)
        test_loss_result = get_loss(model.evaluate(
                np.asarray(X_test),
                np.asarray(y_test)))
        test_loss_result = get_loss(test_loss_result)

    ## save the shape for the next block so we can save it using the definition above
    model_shape = mlp_signature(model)

    print('Current loss {} encoding {}: {}'.format(save_encoding , TRAIN_LOSS, test_loss_result))
    
    results_df = pd.DataFrame([
        {'Encoding Type': f'{ENCODING_TYPE} Encoding', 'Train Loss Metric': TRAIN_LOSS, 'Test Loss': test_loss_result},
        ])

else:
    ## we need to do some fancy allocation of recources if we are going to load both models
    ## do the first on one gpu
    linear_model = load_model(best_model_file_linear, compile=False)
    linear_model.compile(optimizer='adam', loss=TRAIN_LOSS)
    test_loss_result_linear_encoding = get_loss(linear_model.evaluate(
            np.asarray(X_test_linear_encoding),
            np.asarray(y_test_linear_encoding)))
    test_loss_result_linear_encoding = get_loss(test_loss_result_linear_encoding)
    
    ## now do on CPU to avoid gpu allocation
    with tf.device('/CPU:0'):
        onehot_model = load_model(best_model_file_onehot, compile=False)
        onehot_model.compile(optimizer='adam', loss=TRAIN_LOSS)
        test_loss_result_one_hot_encoding = get_loss(onehot_model.evaluate(
                np.asarray(X_test_one_hot_encoding),
                np.asarray(y_test_one_hot_encoding)))
        test_loss_result_one_hot_encoding = get_loss(test_loss_result_one_hot_encoding)

    ## save the shape for the next block so we can save it using the definition above
    model_shape_one_hot_encoding = mlp_signature(onehot_model)
    model_shape_linear_encoding  = mlp_signature(linear_model)
    
    print('Current loss linear encoding {}: {}'.format(TRAIN_LOSS, test_loss_result_linear_encoding))
    print('Current loss one hot encoding {}: {}'.format(TRAIN_LOSS, test_loss_result_one_hot_encoding))
    
    results_df = pd.DataFrame([
        {'Encoding Type': 'Linear Encoding', 'Train Loss Metric': TRAIN_LOSS, 'Test Loss': test_loss_result_linear_encoding},
        {'Encoding Type': 'One Hot Encoding', 'Train Loss Metric': TRAIN_LOSS, 'Test Loss': test_loss_result_one_hot_encoding}
    ])

print(results_df.to_string(index=False))
results_df.to_csv(str(Path(RESULTS_DIR) / 'training/test_loss_results.csv'), index=False)

Measure and print metrics.

## Compare predictions vs. test set

In [ ]:
if 'Try Both' not in ENCODING_TYPE:
    csv_data = [[
        DATA_AUGMENTATION,
        model_shape,
        ENCODING_TYPE,
        test_loss_result,
        TRAIN_LOSS,
        TRAIN_DROPOUT_RATE,
        TRAIN_EARLY_STOPPING_PATIENCE,
        TRAIN_BATCH_SIZE,
        '0.15/0.15',
        LR_INITIAL,
        LR_DECAY_STEPS,
        LR_DECAY_RATE,
        LR_STAIRCASE
        ]]
    
    csv_file = str(Path(RESULTS_DIR) / 'training/history_losses.csv')  ## this doesnt reqrite this file so you need to delete this if you want something fresh
    
    if not os.path.exists(csv_file):
        with open(csv_file, 'w') as file:
            file.write('data_augmentation,model_shape,encoding_type,test_loss,train_loss,train_dropout_rate,train_early_stop_patience,'+
                        'train_batch_size,train_val_split,lr_initial,lr_decay_step,lr_decay_rate,lr_stair_case\n')
    
    with open(csv_file, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerows(csv_data)
    
    ## convert data to DataFrame for easier display
    df = pd.read_csv(csv_file)
    
    def color_red_column(s):
        return ['color: red' if v else '' for v in s]
    
    styled_df = df.style.apply(color_red_column, subset=['test_loss'])
    
    ## display the DataFrame as a table
    display(styled_df)

else:
    ## one hot
    csv_data = [[
        DATA_AUGMENTATION,
        model_shape_one_hot_encoding,
        'One Hot',
        test_loss_result_one_hot_encoding,
        TRAIN_LOSS,
        TRAIN_DROPOUT_RATE,
        TRAIN_EARLY_STOPPING_PATIENCE,
        TRAIN_BATCH_SIZE,
        '0.15/0.15',
        LR_INITIAL,
        LR_DECAY_STEPS,
        LR_DECAY_RATE,
        LR_STAIRCASE
        ]]
    
    csv_file = str(Path(RESULTS_DIR) / 'training/history_losses.csv')  ## this doesnt reqrite this file so you need to delete this if you want something fresh
    
    if not os.path.exists(csv_file):
        with open(csv_file, 'w') as file:
            file.write('data_augmentation,model_shape,encoding_type,test_loss,train_loss,train_dropout_rate,train_early_stop_patience,'+
                        'train_batch_size,train_val_split,lr_initial,lr_decay_step,lr_decay_rate,lr_stair_case\n')
            
    with open(csv_file, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerows(csv_data)

    ## convert data to DataFrame for easier display
    df_one_hot_encoding = pd.read_csv(csv_file)
    
    def color_red_column(s):
        return ['color: red' if v else '' for v in s]
    
    styled_df_one_hot_encoding = df_one_hot_encoding.style.apply(color_red_column, subset=['test_loss'])
    csv_data_linear_encoding = [[
        DATA_AUGMENTATION,
        model_shape_linear_encoding,
        'Linear',
        test_loss_result_linear_encoding,
        TRAIN_LOSS,
        TRAIN_DROPOUT_RATE,
        TRAIN_EARLY_STOPPING_PATIENCE,
        TRAIN_BATCH_SIZE,
        '0.15/0.15',
        LR_INITIAL,
        LR_DECAY_STEPS,
        LR_DECAY_RATE,
        LR_STAIRCASE
        ]]
    
    csv_file_linear_encoding = str(Path(RESULTS_DIR) / 'training/history_losses.csv')  ## this doesnt reqrite this file so you need to delete this if you want something fresh
    
    with open(csv_file_linear_encoding, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerows(csv_data_linear_encoding)
    
    ## convert data to DataFrame for easier display
    df_linear_encoding = pd.read_csv(csv_file_linear_encoding)
    
    def color_red_column(s):
        return ['color: red' if v else '' for v in s]
    
    styled_df_linear_encoding = df_linear_encoding.style.apply(color_red_column, subset=['test_loss'])
    
    ## display the DataFrame as a table
    display(styled_df_linear_encoding)

## decide which model file & test set to use
if 'Try Both' not in ENCODING_TYPE:
    chosen_path = best_model_file  ## whatever var points to the single model file
    X_test_cur = np.asarray(X_test)
    y_test_cur = y_test
else:
    if test_loss_result_linear_encoding < test_loss_result_one_hot_encoding:
        chosen_path = best_model_file_linear
        X_test_cur = np.asarray(X_test_linear_encoding)
        y_test_cur = y_test_linear_encoding
        y_encoding_format_name='linear'
    else:
        chosen_path = best_model_file_onehot
        X_test_cur = np.asarray(X_test_one_hot_encoding)
        y_test_cur = y_test_one_hot_encoding
        y_encoding_format_name='one_hot'

## clear everything again, then load & predict on one cpu device to avoid the memory thing again
tf.keras.backend.clear_session()
gc.collect()

with tf.device('/CPU:0'):
    chosen_model = load_model(chosen_path, compile=False)  ## don't need compile here to just predict
    raw_pred = chosen_model.predict(X_test_cur, verbose=0)

## for multihead models (onehot encoding), raw_pred is a list [continuous_pred, fingers_pred].
## concatenate them back into the same column ordering as y_test so the downstream
## evaluation cells (74, 76) work without any changes.
if 'one hot' in ENCODING_TYPE and 'Try Both' not in ENCODING_TYPE and isinstance(raw_pred, list):
    y_pred = np.concatenate(raw_pred, axis=1)
    print(f"Multi-head prediction: continuous {raw_pred[0].shape} + fingers {raw_pred[1].shape} -> combined {y_pred.shape}")
else:
    y_pred = raw_pred
## now lets look at a specfic case to see how the model predicts things
if 'Try Both' not in ENCODING_TYPE:
    y_encoding_format_name = save_encoding 
    
## load headers from the saved numpy file instead of the csv
headers = np.load(str(Path(METADATA_DIR) / "y_columns.npy"), allow_pickle=True).astype(str).tolist()

X_test_cur = np.asarray(X_test_cur)
y_test_cur = np.asarray(y_test_cur)
y_pred     = np.asarray(y_pred)

n_samples, n_params = y_test_cur.shape
## change nsamples if you don't want to look at everything
n_samples =3

## detect onehot columns for finger_count and prep decoding
fc_base = "design_options.finger_count"
fc_prefix = fc_base + "_"

fc_idx = [j for j, h in enumerate(headers) if h.startswith(fc_prefix)]
fc_class_values = []
for j in fc_idx:
    ## headers like "design_options.finger_count_4"
    try:
        fc_class_values.append(int(headers[j].split(fc_prefix, 1)[1]))
    except Exception:
        fc_class_values.append(None)

## errors for raw percolumn outputs (kept for reference)
sq_errors  = (y_test_cur - y_pred) ** 2
abs_errors = np.abs(y_test_cur - y_pred)

## make a nice dataframe so the output is comprehensible
rows = []
for i in range(n_samples):
    qubit_frequency_GHz, anharmonicity_MHz = X_test_cur[i, 0], X_test_cur[i, 1]

    ## continuous + nonfinger_count onehot outputs
    for j in range(n_params):
        if j in fc_idx:
            continue
        rows.append({
            "sample_idx": i,
            "qubit_frequency_GHz": qubit_frequency_GHz,
            "anharmonicity_MHz": anharmonicity_MHz,
            "param": headers[j],
            "ref": y_test_cur[i, j],
            "pred": y_pred[i, j],
            "abs_error": abs_errors[i, j],
            "sq_error": sq_errors[i, j],
        })

    ## collapsed finger_count (decode by argmax)
    if len(fc_idx) > 0 and all(v is not None for v in fc_class_values):
        ref_vec = y_test_cur[i, fc_idx]
        pred_vec = y_pred[i, fc_idx]

        ref_k = int(np.argmax(ref_vec))
        pred_k = int(np.argmax(pred_vec))

        ref_fc = int(fc_class_values[ref_k])
        pred_fc = int(fc_class_values[pred_k])

        abs_fc = float(abs(ref_fc - pred_fc))
        sq_fc = float((ref_fc - pred_fc) ** 2)

        rows.append({
            "sample_idx": i,
            "qubit_frequency_GHz": qubit_frequency_GHz,
            "anharmonicity_MHz": anharmonicity_MHz,
            "param": fc_base,
            "ref": ref_fc,
            "pred": pred_fc,
            "abs_error": abs_fc,
            "sq_error": sq_fc,
        })

df = pd.DataFrame(rows)

## save it in case we want to do stuff with this in the future
out_csv = Path(RESULTS_DIR) / 'predictions' / f"predictions_and_errors_{y_encoding_format_name}.csv"
df.to_csv(out_csv, index=False, float_format="%.6g")
print(f"\nSaved CSV -> {out_csv.resolve()}\n")

## print it out nicely
for i in range(n_samples):
    sub = df[df["sample_idx"] == i].copy()
    sub = sub[["param", "ref", "pred", "abs_error", "sq_error"]]
    header_line = (
        f"- Sample {i} - "
        f"X: cross_to_ground={X_test_cur[i,0]:.9g}, claw_to_ground={X_test_cur[i,1]:.9g}, cross_to_claw={X_test_cur[i,2]:.9g}, cross_to_cross={X_test_cur[i,3]:.9g}, claw_to_claw={X_test_cur[i,4]:.9g}, ground_to_ground={X_test_cur[i,5]:.9g}"
    )
    print(header_line)
    print(sub.to_string(index=False))
    print() 

## (optional) quick global stats
## (exclude finger_count onehot columns from global stats so they don't skew the averages)
if len(fc_idx) > 0:
    mask = np.ones(n_params, dtype=bool)
    mask[fc_idx] = False
    abs_for_stats = abs_errors[:, mask]
else:
    abs_for_stats = abs_errors

print("Global error stats:")
print("  min abs_error:", float(abs_for_stats.min()))
print("  median abs_error:", float(np.median(abs_for_stats)))
print("  max abs_error:", float(abs_for_stats.max()))

''' 
Here onehot/linear encoding and the mlp which maps categorical data to 1s and 0s is probably 
throwing off the global average. These will be rounded in the future and will probably always 
round to the right number to reconstruct the correct category-- but for now it might throw off 
the overall average error. In the future we might want to just have it consider the non categorical 
data when finding an overall average and reporting that number.
'''

### Unscaled test vs predictions

In [ ]:
## unscale everything and look at errors again.
## you can compare the unscaled actual values to the ml_00...py notebook to convice yourself that unscaling worked

with open(str(Path(METADATA_DIR) / 'X_names'), 'r') as f:
    X_index_names = f.read().splitlines()

## load headers from the saved numpy file (same ordering as y arrays)
headers = np.load(str(Path(METADATA_DIR) / "y_columns.npy"), allow_pickle=True).astype(str).tolist()

## detect onehot columns for finger_count and prep decoding
fc_base = "design_options.finger_count"
fc_prefix = fc_base + "_"

fc_idx = [j for j, h in enumerate(headers) if h.startswith(fc_prefix)]
fc_class_values = []
for j in fc_idx:
    try:
        fc_class_values.append(int(headers[j].split(fc_prefix, 1)[1]))
    except Exception:
        fc_class_values.append(None)

## unscaling x
X_test_unscaled = np.asarray(X_test_cur.copy())
for i in range(X_test_unscaled.shape[0]):
    for j in range(X_test_unscaled.shape[1]):
        scaler = joblib.load(f'{SCALERS_DIR}/scaler_X_{y_encoding_format_name}_{X_index_names[j]}.save')
        X_test_unscaled[i, j] = scaler.inverse_transform([[X_test_unscaled[i, j]]])[0][0]

## unscaling y
y_test_unscaled = np.asarray(y_test_cur.copy())
for i in range(y_test_unscaled.shape[0]):
    for j in range(y_test_unscaled.shape[1]):
        ## onehot columns don't need unscaling (and often no scaler exists / it is identity anyway)
        if j in fc_idx:
            continue
        scaler = joblib.load(f'{SCALERS_DIR}/scaler_y_{y_encoding_format_name}_{headers[j]}.save')
        y_test_unscaled[i, j] = scaler.inverse_transform([[y_test_unscaled[i, j]]])[0][0]

## unscaling y predictions
y_pred_unscaled = np.asarray(y_pred.copy())
for i in range(y_pred_unscaled.shape[0]):
    for j in range(y_pred_unscaled.shape[1]):
        if j in fc_idx:
            continue
        scaler = joblib.load(f'{SCALERS_DIR}/scaler_y_{y_encoding_format_name}_{headers[j]}.save')
        y_pred_unscaled[i, j] = scaler.inverse_transform([[y_pred_unscaled[i, j]]])[0][0]

n_samples, n_params = y_test_unscaled.shape
n_samples = 3

## find how good or bad we did (the errors)
sq_errors_unscaled  = (y_test_unscaled - y_pred_unscaled) ** 2
abs_errors_unscaled = np.abs(y_test_unscaled - y_pred_unscaled)

## making a nice fancy dataframe, we like fancy things
rows_unscaled = []
for i in range(n_samples):
    qubit_frequency_GHz, anharmonicity_MHz = X_test_unscaled[i, 0], X_test_unscaled[i, 1]

    ## continuous + nonfinger_count onehot outputs
    for j in range(n_params):
        if j in fc_idx:
            continue
        rows_unscaled.append({
            "sample_idx": i,
            "qubit_frequency_GHz": qubit_frequency_GHz,
            "anharmonicity_MHz": anharmonicity_MHz,
            "param": headers[j],
            "ref_unscaled": y_test_unscaled[i, j],
            "pred_unscaled": y_pred_unscaled[i, j],
            "abs_error_unscaled": abs_errors_unscaled[i, j],
            "sq_error_unscaled": sq_errors_unscaled[i, j],
        })

    ## collapsed finger_count (decode by argmax)
    if len(fc_idx) > 0 and all(v is not None for v in fc_class_values):
        ref_vec = y_test_unscaled[i, fc_idx]
        pred_vec = y_pred_unscaled[i, fc_idx]

        ref_k = int(np.argmax(ref_vec))
        pred_k = int(np.argmax(pred_vec))

        ref_fc = int(fc_class_values[ref_k])
        pred_fc = int(fc_class_values[pred_k])

        abs_fc = float(abs(ref_fc - pred_fc))
        sq_fc = float((ref_fc - pred_fc) ** 2)

        rows_unscaled.append({
            "sample_idx": i,
            "qubit_frequency_GHz": qubit_frequency_GHz,
            "anharmonicity_MHz": anharmonicity_MHz,
            "param": fc_base,
            "ref_unscaled": ref_fc,
            "pred_unscaled": pred_fc,
            "abs_error_unscaled": abs_fc,
            "sq_error_unscaled": sq_fc,
        })

df_unscaled = pd.DataFrame(rows_unscaled)

## save csv of unscaled results uncase we lose this notebook due to github blowing up, ya never know
out_csv_unscaled = Path(RESULTS_DIR) / 'predictions' / f"predictions_and_errors_unscaled_{y_encoding_format_name}.csv"
df_unscaled.to_csv(out_csv_unscaled, index=False, float_format="%.6g")
print(f"\nSaved CSV -> {out_csv_unscaled.resolve()}\n")

## print out stuff so you can see it here if you are to lazy like me to open a csv
for i in range(n_samples):
    sub = df_unscaled[df_unscaled["sample_idx"] == i].copy()
    sub = sub[["param", "ref_unscaled", "pred_unscaled", "abs_error_unscaled", "sq_error_unscaled"]]
    header_line = (
        f"- Sample {i} (Unscaled) - "
        f"X: cross_to_ground={X_test_unscaled[i,0]:.9g}, claw_to_ground={X_test_unscaled[i,1]:.9g}, cross_to_claw={X_test_unscaled[i,2]:.9g}, cross_to_cross={X_test_unscaled[i,3]:.9g}, claw_to_claw={X_test_unscaled[i,4]:.9g}, ground_to_ground={X_test_unscaled[i,5]:.9g}"
    )
    print(header_line)
    print(sub.to_string(index=False))
    print()

## look at overall stats, see below comment for a caviat
## (exclude finger_count onehot columns from global stats so they don't skew the averages)
if len(fc_idx) > 0:
    mask = np.ones(n_params, dtype=bool)
    mask[fc_idx] = False
    abs_for_stats = abs_errors_unscaled[:, mask]
else:
    abs_for_stats = abs_errors_unscaled

print("Global unscaled error stats:")
print("  min abs_error:", float(abs_for_stats.min()))
print("  median abs_error:", float(np.median(abs_for_stats)))
print("  max abs_error:", float(abs_for_stats.max()))

'''
Here onehot/linear encoding and the MLP which maps categorical data to 1s and 0s is probably 
throwing off the global average. These will be rounded in the future and will probably always 
round to the right number to reconstruct the correct category-- but for now it might throw off 
the overall average error. In the future we might want to just have it consider the non-categorical 
data when finding an overall average and reporting that number.
'''